In [4]:
import numpy as np
import pandas as pd
from sklearn.utils import shuffle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim


Train_Data_True=pd.read_csv("C:/Users/Lenovo/OneDrive/Documents/True.csv").head(2000)
Train_Data_Fake=pd.read_csv("C:/Users/Lenovo/OneDrive/Documents/Fake.csv").head(2000)
Train_Data_Fake['label']=0
Train_Data_True['label']=1
Train_Data=pd.concat([Train_Data_Fake,Train_Data_True],ignore_index=True)
Train_Data = shuffle(Train_Data, random_state=42)

X=Train_Data.drop(columns=['label','date'])
y=Train_Data['label']

X["content"] = (
    X["title"].fillna("").astype(str) + " " +
    X["text"].fillna("").astype(str) + " " +
    X["subject"].fillna("").astype(str)
)

vectorizer=TfidfVectorizer(ngram_range=(1,2))
X=vectorizer.fit_transform(X['content'])

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

X_train=torch.tensor(X_train.toarray(),dtype=torch.float32)
X_test=torch.tensor(X_test.toarray(),dtype=torch.float32)
y_train=torch.tensor(y_train,dtype=torch.long)
y_test=torch.tensor(y_test.to_numpy(),dtype=torch.long)

class FakeNewsDetection(nn.Module):
    def __init__(self,input_size):
        super().__init__()
        self.fc1=nn.Linear(input_size,128)
        self.relu1=nn.ReLU()
        self.fc2=nn.Linear(128,64)
        self.relu2=nn.ReLU()
        self.fc3=nn.Linear(64,32)
        self.relu3=nn.ReLU()
        self.fc4=nn.Linear(32,2)
    def forward(self,x):
        x=self.fc1(x)
        x=self.relu1(x)
        x=self.fc2(x)
        x=self.relu2(x)
        x=self.fc3(x)
        x=self.relu3(x)
        x=self.fc4(x)
        return x
    
input_size=X_train.shape[1]
model=FakeNewsDetection(input_size)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=0.001)

epochs=100
for epoch in range(epochs):
    model.train()
    output=model(X_train)
    loss=criterion(output,y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    output=model(X_test)
    prediction=torch.argmax(output,dim=1)
    accuracy = accuracy_score(y_test.numpy(), prediction.numpy())
print(accuracy)

## Prediction

new_text=[
    'Scientists have announced that drinking five cups of coffee every day makes people live up to 150 years. The discovery has been confirmed by every major hospital worldwide, and governments are planning to make coffee mandatory for all citizens starting next month.']
text_vector=vectorizer.transform(new_text)
text_tensor=torch.tensor(text_vector.toarray(),dtype=torch.float32)
model.eval()
with torch.no_grad():

    prediction = model(text_tensor)

    predicted_class = torch.argmax(prediction, dim=1)
if predicted_class.item() == 0:
    print("Fake News")
else:
    print("True News")




Epoch 10/100, Loss: 0.6143
Epoch 20/100, Loss: 0.3973
Epoch 30/100, Loss: 0.1388
Epoch 40/100, Loss: 0.0168
Epoch 50/100, Loss: 0.0015
Epoch 60/100, Loss: 0.0004
Epoch 70/100, Loss: 0.0002
Epoch 80/100, Loss: 0.0002
Epoch 90/100, Loss: 0.0001
Epoch 100/100, Loss: 0.0001
0.985
Fake News
